# Simple log(gf) tweak and line-profile fit

This notebook keeps things minimal:
1) change log(gf) and measure delta flux
2) fit log(gf) to a provided line profile

Note: VALD linelist wavelengths are vacuum. We pick the line center from the linelist to avoid air/vacuum mismatches.

This version loads the observed solar spectrum from `jorg/data/obs_spec.txt` and interpolates it onto the synthesis grid.


In [1]:
import sys
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
from scipy.optimize import minimize

# Find repo root by walking up from notebook location
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / 'jorg' / 'src').exists() and (repo_root / 'data').exists():
        break
    repo_root = repo_root.parent
else:
    raise RuntimeError('Could not find Korg.jl repo root. Expected jorg/src and data directories.')

sys.path.insert(0, str(repo_root / 'jorg' / 'src'))

from jorg.atmosphere import interpolate_marcs
from jorg.synthesis import (
    create_korg_compatible_abundance_array,
    synthesize,
)
from jorg.lines.linelist import read_linelist
from jorg.lines.linelist_modifier import LogGFModifier
from jorg.utils.wavelength_utils import air_to_vacuum


In [2]:
# Basic setup
Teff = 5780
logg = 4.44
m_H = 0.0
vmic = 1.0
hydrogen_lines = False
cntm_step = 1.0

atm = interpolate_marcs(Teff, logg, m_H)
A_X = create_korg_compatible_abundance_array(m_H)

# Use a small wavelength window to keep this fast
wl_min, wl_max = 5000.0, 5200.0

linelist_path = repo_root / 'data' / 'linelists' / 'vald_extract_stellar_solar_threshold001.vald'
linelist = read_linelist(str(linelist_path), format='vald')

# Pick a line close to your target wavelength
target_wl = 5001.6
line_wls = linelist.wavelengths_angstrom()
line_idx = int(np.argmin(np.abs(line_wls - target_wl)))
line_center = line_wls[line_idx]
line = linelist[line_idx]
line_label = f"{line.species} @ {line_center:.4f} A"
print(f'Using line_center = {line_center:.4f} Angstrom')

# Synthesize spectrum in one call (lines + continuum)
base_result = synthesize(
    atm,
    linelist,
    A_X,
    wavelengths=(wl_min, wl_max),
    hydrogen_lines=hydrogen_lines,
    verbose=False,
    logg=logg,
    cntm_step=cntm_step,
    vmic=vmic,
)

wavelength_grid = np.asarray(base_result.wavelengths)
base_flux = np.asarray(base_result.flux) / np.asarray(base_result.cntm)

# Load observed solar spectrum and interpolate onto our grid
obs_path = repo_root / 'jorg' / 'data' / 'obs_spec.txt'
obs_lines = obs_path.read_text().splitlines()
obs_wl = np.fromstring(obs_lines[0], sep=' ')
obs_flux_raw = np.fromstring(obs_lines[1], sep=' ')
finite = np.isfinite(obs_flux_raw)
obs_wl = obs_wl[finite]
obs_flux_raw = obs_flux_raw[finite]

# obs_spec.txt is in air wavelengths; convert to vacuum to match VALD/Jorg
obs_wavelengths = 'air'
if obs_wavelengths == 'air':
    obs_wl = air_to_vacuum(obs_wl * 1e-8) * 1e8

obs_cont = np.nanpercentile(obs_flux_raw, 99.5)
obs_flux = obs_flux_raw / obs_cont
observed_flux = np.interp(wavelength_grid, obs_wl, obs_flux)
print(f'Observed spectrum: {len(obs_wl)} points, continuum~{obs_cont:.3f} ({obs_wavelengths}->vacuum)')


📖 Reading linelist: vald_extract_stellar_solar_threshold001.vald
   Format: vald
   Wavelength unit: auto
   Detected vacuum wavelengths - no conversion needed
   Found 41880 data lines
   Successfully parsed 41861 lines
   After Korg.jl filtering: 41861 lines (removed 0 lines)
Using line_center = 5001.6058 Angstrom
Loaded 273 EXACT partition functions from Korg.jl
  Added bare nuclei: H II, He III (U=1)
Observed spectrum: 165853 points, continuum~1.024 (air->vacuum)


In [5]:
# Change log(gf) and compute delta flux
delta_loggf = 0.5

modifier = LogGFModifier(linelist, wavelength_tolerance=0.01)
modifier.adjust_line(line_center, delta_loggf=delta_loggf)
modified_linelist = modifier.apply_modifications()

modified_result = synthesize(
    atm,
    modified_linelist,
    A_X,
    wavelengths=(wl_min, wl_max),
    hydrogen_lines=hydrogen_lines,
    verbose=False,
    logg=logg,
    cntm_step=cntm_step,
    vmic=vmic,
    use_chemical_equilibrium_from=base_result,
)

modified_flux = np.asarray(modified_result.flux) / np.asarray(modified_result.cntm)
delta_flux = modified_flux - base_flux
print(f'Delta flux: min={delta_flux.min():.3e}, max={delta_flux.max():.3e}')

fig = go.Figure()
fig.add_trace(go.Scatter(x=wavelength_grid, y=base_flux, mode='lines', name='base'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=modified_flux, mode='lines', name='modified'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=delta_flux, mode='lines', name='delta', line=dict(dash='dash')))
fig.add_vline(x=line_center, line_dash='dot', line_color='gray', line_width=1)
fig.add_annotation(
    x=line_center,
    y=float(np.max(base_flux)),
    text=line_label,
    showarrow=False,
    textangle=90,
    xanchor='left',
    yanchor='bottom'
)
fig.update_layout(
    template='plotly_white',
    width=800,
    height=400,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Normalized flux',
)
fig.show()


Delta flux: min=-2.295e-01, max=-1.375e-05


NameError: name 'go' is not defined

In [4]:
# Fit log(gf) to a given line profile using BFGS
# observed_flux is loaded from obs_spec.txt

def objective(x):
    delta = float(x[0])
    modifier = LogGFModifier(linelist, wavelength_tolerance=0.01)
    modifier.adjust_line(line_center, delta_loggf=delta)
    candidate_linelist = modifier.apply_modifications()

    result = synthesize(
        atm,
        candidate_linelist,
        A_X,
        wavelengths=(wl_min, wl_max),
        hydrogen_lines=hydrogen_lines,
        verbose=False,
        logg=logg,
        cntm_step=cntm_step,
        vmic=vmic,
        use_chemical_equilibrium_from=base_result,
    )
    model_wl = np.asarray(result.wavelengths)
    model_flux = np.asarray(result.flux)
    if model_wl.shape != wavelength_grid.shape or not np.allclose(model_wl, wavelength_grid):
        model_flux = np.interp(wavelength_grid, model_wl, model_flux)
    model_flux = model_flux / np.asarray(result.cntm)
    return float(np.mean((model_flux - observed_flux) ** 2))

x0 = np.array([0.0])
opt = minimize(objective, x0, method='BFGS', options={'gtol': 1e-4, 'maxiter': 20})
best_delta = float(opt.x[0])
print(f'Best delta_loggf = {best_delta:.3f} (success={opt.success})')

best_modifier = LogGFModifier(linelist, wavelength_tolerance=0.01)
best_modifier.adjust_line(line_center, delta_loggf=best_delta)
best_result = synthesize(
    atm,
    best_modifier.apply_modifications(),
    A_X,
    wavelengths=(wl_min, wl_max),
    hydrogen_lines=hydrogen_lines,
    verbose=False,
    logg=logg,
    cntm_step=cntm_step,
    vmic=vmic,
    use_chemical_equilibrium_from=base_result,
)
best_flux = np.asarray(best_result.flux) / np.asarray(best_result.cntm)

fig = go.Figure()
fig.add_trace(go.Scatter(x=wavelength_grid, y=observed_flux, mode='lines', name='observed'))
fig.add_trace(go.Scatter(x=wavelength_grid, y=best_flux, mode='lines', name='best-fit'))
fig.add_vline(x=line_center, line_dash='dot', line_color='gray', line_width=1)
fig.add_annotation(
    x=line_center,
    y=float(np.max(observed_flux)),
    text=line_label,
    showarrow=False,
    textangle=90,
    xanchor='left',
    yanchor='bottom'
)
fig.update_layout(
    template='plotly_white',
    width=800,
    height=400,
    xaxis_title='Wavelength (Angstrom)',
    yaxis_title='Normalized flux',
)
fig.show()


Best delta_loggf = 0.000 (success=True)
